# 01. Echoes·FMA 데이터 품질 점검

연구 주제는 **미지 생성기와 오디오 압축 환경에서의 강건한 AI 생성 음악 탐지**이다. 모델 학습에 앞서 Echoes manifest, 실제 오디오 파일, FMA metadata의 구조와 누락 여부를 확인했다.

### 점검 결과

| 항목 | 결과 |
|---|---:|
| Echoes manifest | 4,468행 |
| TTA / ATA | 3,165 / 1,303행 |
| 생성기 / 장르 | 12종 / 3종 |
| TTA 경로 충돌 | 3행 |
| Clean TTA | 3,162행 |
| `original_audio` | 296개 |
| Clean TTA 누락 파일 | 0개 |
| 실제 오디오 / manifest 고유 경로 | 4,488 / 4,464개 |
| Manifest 밖 오디오 | 24개 |

분석에는 Echoes의 TTA만 사용하고 ATA와 manifest에 없는 파일은 제외한다. 이후 데이터 분할은 같은 원곡에서 나온 파일이 서로 다른 split에 들어가지 않도록 `original_audio` 단위로 수행한다.


## 0. 경로 설정

Echoes manifest와 FMA `tracks.csv`의 위치를 프로젝트 루트 기준으로 지정한다.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

ECHOES_ROOT = PROJECT_ROOT / "data/raw/Echoes/Echoes"
ECHOES_MANIFEST = ECHOES_ROOT / "dataset_manifest.csv"

FMA_META_ROOT = PROJECT_ROOT / "data/raw/FMA/fma_metadata"
FMA_TRACKS = FMA_META_ROOT / "tracks.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Echoes manifest exists:", ECHOES_MANIFEST.exists())
print("FMA tracks.csv exists:", FMA_TRACKS.exists())


PROJECT_ROOT: <PROJECT_ROOT>
Echoes manifest exists: True
FMA tracks.csv exists: True


**결과:** 프로젝트 루트가 현재 작업 폴더로 설정되었고, Echoes manifest와 FMA `tracks.csv`가 모두 존재했다.

## 1. Echoes manifest 구조

Manifest를 불러와 행·열 수, 컬럼 이름, 앞의 3개 행을 확인한다.


In [2]:
echoes = pd.read_csv(ECHOES_MANIFEST)

print("shape:", echoes.shape)
print("\ncolumns:")
print(echoes.columns.tolist())

display(echoes.head(3))


shape: (4468, 7)

columns:
['path_in_dataset', 'original_audio', 'generator', 'type', 'genre', 'description', 'duration']


,path_in_dataset,original_audio,generator,type,genre,description,duration
0,TTA/acestep/10000_People_Chanting_Im_an_Indivi...,"10,000 People Chanting, ""I'm an Individual"" - ...",acestep,TTA,Electronic,"cinematic, idm, downtempo, layered, swelling, ...",45.70
1,TTA/acestep/1984_Punk_Rock_Opera_acestep_TTA_0...,1984 - Punk Rock Opera,acestep,TTA,Rock,"hardcore-punk, political, d-beat, shouted-chor...",54.80
2,TTA/acestep/2Much_Andy_Spinelli_Alex_Sánchez_H...,2Much (Andy Spinelli & Alex Sánchez House Edit...,acestep,TTA,Electronic,"house, four-on-the-floor, piano-stabs, filtere...",80.71


**결과:** Echoes manifest는 4,468행, 7열이다. 컬럼은 파일 경로, 원곡명, 생성기, 생성 유형, 장르, description, 재생시간이며 앞의 3개 행도 같은 구조로 읽혔다.


In [3]:
print("===== TYPE =====")
print(echoes["type"].value_counts(dropna=False))

print("\n===== GENERATOR =====")
print(echoes["generator"].value_counts(dropna=False))

print("\n===== GENRE =====")
print(echoes["genre"].value_counts(dropna=False))


===== TYPE =====
type
TTA    3165
ATA    1303
Name: count, dtype: int64

===== GENERATOR =====
generator
diffrhythm     594
musicgen       591
audioldm       587
songgen        561
producer       300
suno           300
udio           300
elevenlabs     300
brev           298
acestep        294
stableaudio    194
mubert         149
Name: count, dtype: int64

===== GENRE =====
genre
Electronic    1653
Rock          1594
Pop           1221
Name: count, dtype: int64


**결과:** 전체 4,468행은 TTA 3,165행과 ATA 1,303행으로 구성된다. 생성기는 12종이며 장르는 Electronic 1,653행, Rock 1,594행, Pop 1,221행이다.


## 2. TTA 데이터 확인

연구 범위에 해당하는 TTA만 분리해 원곡, 생성기, 장르와 재생시간 분포를 살펴본다.


In [4]:
tta = echoes[echoes["type"] == "TTA"].copy()

print("TTA rows:", len(tta))
print("Unique original_audio:", tta["original_audio"].nunique())
print("Unique generators:", tta["generator"].nunique())

print("\n===== TTA GENERATOR =====")
print(tta["generator"].value_counts())

print("\n===== TTA GENRE =====")
print(tta["genre"].value_counts())

print("\n===== DURATION =====")
print(tta["duration"].describe())


TTA rows: 3165
Unique original_audio: 296
Unique generators: 12

===== TTA GENERATOR =====
generator
suno           300
udio           300
elevenlabs     300
diffrhythm     299
brev           298
musicgen       296
acestep        294
audioldm       292
songgen        292
stableaudio    194
producer       151
mubert         149
Name: count, dtype: int64

===== TTA GENRE =====
genre
Electronic    1131
Rock          1130
Pop            904
Name: count, dtype: int64

===== DURATION =====
count    3165.000000
mean      121.362408
std        74.902808
min        17.960000
25%        30.720000
50%       130.860000
75%       180.000000
max       479.960000
Name: duration, dtype: float64


**결과:** TTA는 3,165행이며 `original_audio` 296개와 생성기 12종을 포함한다. 장르는 Electronic 1,131행, Rock 1,130행, Pop 904행이고, 재생시간은 평균 121.36초(최소 17.96초, 최대 479.96초)였다.

## 3. TTA 경로 중복 검사

같은 `path_in_dataset`이 서로 다른 manifest 행에 연결된 경우를 찾는다.


In [5]:
tta_dup = (
    tta[tta["path_in_dataset"].duplicated(keep=False)]
    .sort_values("path_in_dataset")
)

print("Duplicated TTA rows:", len(tta_dup))
print("Duplicated TTA unique paths:", tta_dup["path_in_dataset"].nunique())

display(
    tta_dup[
        ["path_in_dataset", "original_audio", "generator", "genre"]
    ]
)


Duplicated TTA rows: 3
Duplicated TTA unique paths: 1


,path_in_dataset,original_audio,generator,genre
4454,TTA/musicgen/_musicgen_TTA_001.wav,В Наших Сердцах - Чокнутый Пропеллер,musicgen,Rock
4456,TTA/musicgen/_musicgen_TTA_001.wav,Глазами Детей - Чокнутый Пропеллер,musicgen,Rock
4458,TTA/musicgen/_musicgen_TTA_001.wav,Кортни Лав - Чокнутый Пропеллер,musicgen,Rock


**결과:** 중복된 TTA 경로는 `TTA/musicgen/_musicgen_TTA_001.wav` 하나이며, 서로 다른 원곡 3개가 이 파일을 공유한다. 실제 대응 원곡을 정할 수 없어 세 행을 모두 제외한다.


## 4. Clean TTA 생성과 파일 확인

경로가 충돌한 세 행을 제거하고, 남은 manifest 경로가 실제 파일로 존재하는지 확인한다.


In [6]:
dup_mask = tta["path_in_dataset"].duplicated(keep=False)
tta_clean = tta[~dup_mask].copy()

tta_clean["full_path"] = tta_clean["path_in_dataset"].apply(
    lambda x: ECHOES_ROOT / x
)
tta_clean["exists"] = tta_clean["full_path"].apply(lambda p: p.exists())

print("===== CLEAN TTA =====")
print("Original TTA rows :", len(tta))
print("Excluded rows     :", int(dup_mask.sum()))
print("Clean TTA rows    :", len(tta_clean))
print("Generators        :", tta_clean["generator"].nunique())
print("Original groups   :", tta_clean["original_audio"].nunique())

print("\n===== GENERATOR COUNTS =====")
print(tta_clean["generator"].value_counts())

print("\n===== GENRE COUNTS =====")
print(tta_clean["genre"].value_counts())

print("\n===== FILE CHECK =====")
print("Existing files :", int(tta_clean["exists"].sum()))
print("Missing files  :", int((~tta_clean["exists"]).sum()))


===== CLEAN TTA =====
Original TTA rows : 3165
Excluded rows     : 3
Clean TTA rows    : 3162
Generators        : 12
Original groups   : 296

===== GENERATOR COUNTS =====
generator
suno           300
udio           300
elevenlabs     300
diffrhythm     299
brev           298
acestep        294
musicgen       293
audioldm       292
songgen        292
stableaudio    194
producer       151
mubert         149
Name: count, dtype: int64

===== GENRE COUNTS =====
genre
Electronic    1131
Rock          1127
Pop            904
Name: count, dtype: int64

===== FILE CHECK =====
Existing files : 3162
Missing files  : 0


**결과:** 3,165행에서 충돌 3행을 제외해 Clean TTA 3,162행을 얻었다. `original_audio`는 296개, 생성기는 12종이며 실제 파일은 3,162개 모두 존재한다.

생성기별 행 수는 Suno·Udio·ElevenLabs가 각 300개로 가장 많고, Mubert가 149개로 가장 적다. 장르별로는 Electronic 1,131개, Rock 1,127개, Pop 904개다.


## 5. Manifest와 실제 오디오 비교

Manifest에 적힌 경로 집합과 폴더에서 찾은 WAV·MP3·FLAC 파일 집합을 비교한다.


In [7]:
manifest_paths = set(
    echoes["path_in_dataset"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
)

audio_exts = {".wav", ".mp3", ".flac"}
actual_paths = set()

for p in ECHOES_ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in audio_exts:
        actual_paths.add(p.relative_to(ECHOES_ROOT).as_posix())

extra_files = sorted(actual_paths - manifest_paths)
missing_files = sorted(manifest_paths - actual_paths)

print("===== MANIFEST vs ACTUAL AUDIO =====")
print("Manifest rows         :", len(echoes))
print("Unique manifest paths :", len(manifest_paths))
print("Actual audio files    :", len(actual_paths))
print("Extra audio files     :", len(extra_files))
print("Missing audio files   :", len(missing_files))

print("\n===== EXTRA FILES =====")
for x in extra_files:
    print(x)


===== MANIFEST vs ACTUAL AUDIO =====
Manifest rows         : 4468
Unique manifest paths : 4464
Actual audio files    : 4488
Extra audio files     : 24
Missing audio files   : 0

===== EXTRA FILES =====
ATA/songgen/1984_Punk_Rock_Opera_songgen_ATA_001.mp3
ATA/songgen/2_Wasnt_There_Isle_of_Pine_songgen_ATA_001.mp3
ATA/songgen/50000_Volts_of_Democracy_mp3_Legally_Blind_songgen_ATA_001.mp3
ATA/songgen/Attention_Pete_Prodoehl_songgen_ATA_001.mp3
ATA/songgen/Bergwald_Bergwald_Every_Now_and_Every_Then_songgen_ATA_001.mp3
ATA/songgen/Fear_Los_Fancy_Free_songgen_ATA_001.mp3
ATA/songgen/Five_40_DerbySlow_Whistle_The_Crypts_songgen_ATA_001.mp3
ATA/songgen/Gloomy_Sunday_AmortE_songgen_ATA_001.mp3
ATA/songgen/I_Dream_So_Vividly_Uninhabitable_Mansions_songgen_ATA_001.mp3
ATA/songgen/KISS_my_Boots_The_Zombie_Dandies_songgen_ATA_001.mp3
ATA/songgen/Leaving_Here_Mod_Fun_songgen_ATA_001.mp3
ATA/songgen/March_of_the_GPA_Mechanics_tghost_songgen_ATA_001.mp3
ATA/songgen/Mettle_Pipe_Choir_songgen_ATA_001.mp

**결과:** manifest 4,468행에는 고유 경로가 4,464개 있고, 실제 오디오 파일은 4,488개다. Manifest 경로 중 누락된 파일은 없으며, 반대로 manifest에 없는 파일은 24개(ATA/SongGen 23개, TTA/ACE-Step 1개)다. 이 24개는 metadata 연결을 확인할 수 없어 분석에서 제외한다.


## 6. 전체 manifest 경로 중복 검사

TTA와 ATA를 합친 manifest 전체에서 반복되는 파일 경로를 확인한다.


In [8]:
all_dup = (
    echoes[echoes["path_in_dataset"].duplicated(keep=False)]
    .sort_values("path_in_dataset")
)

print("Duplicated rows        :", len(all_dup))
print("Duplicated unique paths:", all_dup["path_in_dataset"].nunique())

display(
    all_dup[
        ["path_in_dataset", "original_audio", "generator", "type", "genre"]
    ]
)


Duplicated rows        : 6
Duplicated unique paths: 2


,path_in_dataset,original_audio,generator,type,genre
4455,ATA/musicgen/_musicgen_ATA_001.wav,В Наших Сердцах - Чокнутый Пропеллер,musicgen,ATA,Rock
4457,ATA/musicgen/_musicgen_ATA_001.wav,Глазами Детей - Чокнутый Пропеллер,musicgen,ATA,Rock
4459,ATA/musicgen/_musicgen_ATA_001.wav,Кортни Лав - Чокнутый Пропеллер,musicgen,ATA,Rock
4454,TTA/musicgen/_musicgen_TTA_001.wav,В Наших Сердцах - Чокнутый Пропеллер,musicgen,TTA,Rock
4456,TTA/musicgen/_musicgen_TTA_001.wav,Глазами Детей - Чокнутый Пропеллер,musicgen,TTA,Rock
4458,TTA/musicgen/_musicgen_TTA_001.wav,Кортни Лав - Чокнутый Пропеллер,musicgen,TTA,Rock


**결과:** 중복 경로는 ATA와 TTA의 MusicGen 파일 각 1개로, 모두 세 원곡 행에서 반복된다. 따라서 중복 행은 총 6행이고 고유 중복 경로는 2개다.


## 7. FMA metadata 구조

FMA `tracks.csv`를 2단계 컬럼 헤더로 읽고 매칭에 필요한 컬럼을 확인한다.


In [9]:
fma = pd.read_csv(
    FMA_TRACKS,
    header=[0, 1],
    index_col=0
)

print("FMA shape:", fma.shape)
print("\nFirst 50 columns:")
print(fma.columns.tolist()[:50])

display(fma.head(3))


FMA shape: (106574, 52)

First 50 columns:
[('album', 'comments'), ('album', 'date_created'), ('album', 'date_released'), ('album', 'engineer'), ('album', 'favorites'), ('album', 'id'), ('album', 'information'), ('album', 'listens'), ('album', 'producer'), ('album', 'tags'), ('album', 'title'), ('album', 'tracks'), ('album', 'type'), ('artist', 'active_year_begin'), ('artist', 'active_year_end'), ('artist', 'associated_labels'), ('artist', 'bio'), ('artist', 'comments'), ('artist', 'date_created'), ('artist', 'favorites'), ('artist', 'id'), ('artist', 'latitude'), ('artist', 'location'), ('artist', 'longitude'), ('artist', 'members'), ('artist', 'name'), ('artist', 'related_projects'), ('artist', 'tags'), ('artist', 'website'), ('artist', 'wikipedia_page'), ('set', 'split'), ('set', 'subset'), ('track', 'bit_rate'), ('track', 'comments'), ('track', 'composer'), ('track', 'date_created'), ('track', 'date_recorded'), ('track', 'duration'), ('track', 'favorites'), ('track', 'genre_top'), 

album                                                     \
         comments         date_created        date_released engineer   
track_id                                                               
2               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
3               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
5               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   

                                                         ...       track  \
         favorites id information listens producer tags  ... information   
track_id                                                 ...               
2                4  1     <p></p>    6073      NaN   []  ...         NaN   
3                4  1     <p></p>    6073      NaN   []  ...         NaN   
5                4  1     <p></p>    6073      NaN   []  ...         NaN   

                                 \
         interest language_code   
track_id                          
2            4656            en   
3            1470            en   
5            1933            en   

                                                                              \
                                                    license listens lyricist   
track_id                                                                       
2         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1293      NaN   
3         Attribution-NonCommercial-ShareAlike 3.0 Inter...     514      NaN   
5         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1151      NaN   

                                              
         number publisher tags         title  
track_id                                      
2             3       NaN   []          Food  
3             4       NaN   []  Electric Ave  
5             6       NaN   []    This World  

[3 rows x 52 columns]

**결과:** FMA `tracks.csv`는 106,574행, 52열이다. 이후 매칭에는 아티스트명, 곡 제목, 대표 장르, 라이선스, 재생시간, subset 정보를 사용한다.


## 8. 정제 기준과 다음 작업

Echoes 4,468행에서 ATA를 제외하고 TTA 경로 충돌 3행을 제거해 FAKE 데이터 3,162개를 확정했다. 이 데이터는 296개 원곡 그룹과 12개 생성기로 구성되며 파일 누락은 없다.

다음 노트북 `02_fma_matching_check.ipynb`에서는 296개 `original_audio`를 FMA의 제목·아티스트 정보와 매칭하고, 복수 후보를 정리해 REAL track 목록을 만든다.


## 정리

- Echoes manifest: 4,468행(TTA 3,165행, ATA 1,303행)
- Clean TTA: 3,162행, `original_audio` 296개
- Clean TTA 누락 파일: 0개
- Manifest 밖 오디오: 24개(분석 제외)
- FMA metadata: 106,574행, 52열
